# TBBT - adnotacje gotowe

Składa zweryfikowane pliki adnotatora w rejestr zbiorczy, wylicza zakresy odcinków wchodzące do korpusu i buduje pliki zapytań. Odcinki jeszcze niezweryfikowane pomija i wypisuje, więc można go uruchamiać w miarę postępu.

**Wymaga:** `tbbt_02_annotation_candidates.ipynb` oraz przejrzenia plików `data/annotations/tbbt/*_intervals.csv` w `tools/interval-annotator`.

**Zapisuje:**

| plik | co zawiera |
|---|---|
| `data/interim/tbbt/tbbt_annotations.csv` | rejestr zbiorczy zaktualizowany o decyzje ręczne; historia decyzji jest przenoszona |
| `data/interim/tbbt/tbbt_ranges.csv` | przedziały odcinków wchodzące do korpusu, czyta to segmentacja |
| `data/annotations/tbbt/tbbt_query_tags.csv` | znaczniki wymagań, złożoność i postacie; dopisywane są tylko brakujące wiersze |
| `data/annotations/tbbt/tbbt_queries_dev.jsonl` | zapytania zbioru deweloperskiego |
| `data/annotations/tbbt/tbbt_queries_test.jsonl` | zapytania zbioru testowego |

**Dalej:** `tbbt_04_profiles.ipynb`, a potem eksperymenty.

In [ ]:
SERIES        = "tbbt"
EPISODE_COUNT = 24       # reads tbbt_selection_<COUNT>.csv
REQUIRE_ALL   = False    # True = stop unless every episode is verified

import csv
import importlib
import re
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.annotation import build, tags
from src.annotation import intervals as iv
from src.annotation import ranges as rg
from src.annotation import registry as reg
from src.segmentation.timeline import Timeline
from src.utils import settings

for module in (iv, rg, reg, tags, build, settings):
    importlib.reload(module)      # the kernel keeps a once-imported module in memory

DATA_DIR        = ROOT / "data" / "interim" / SERIES
WORK_DIR        = DATA_DIR / "work"
ANNOTATIONS_DIR = ROOT / "data" / "annotations" / SERIES

REGISTER_CSV = DATA_DIR / f"{SERIES}_annotations.csv"
RANGES_CSV   = DATA_DIR / f"{SERIES}_ranges.csv"
HOLES_CSV    = DATA_DIR / f"{SERIES}_holes.csv"
TAGS_CSV     = tags.path_for(SERIES, ANNOTATIONS_DIR)
TAGS_NEW     = TAGS_CSV.with_name(TAGS_CSV.stem + "_new.csv")

MIN_DURATION, MAX_DURATION = settings.MIN_DURATION, settings.MAX_DURATION
MIN_RANGE = rg.MIN_RANGE
PROFILED_CHARACTERS = settings.profiled(SERIES)
SPLITS = settings.SPLITS


def mmss(seconds):
    """Seconds as minutes:seconds, the way a player shows a position."""
    total = round(seconds)
    return f"{total // 60}:{total % 60:02d}"


def hhmmss(seconds):
    """Seconds as hours:minutes:seconds, for the sums."""
    total = round(seconds)
    return f"{total // 3600}:{total // 60 % 60:02d}:{total % 60:02d}"


def duration_cell(seconds, clock=mmss, width=18):
    """Seconds with the clock reading next to them, right-aligned."""
    return f"{seconds:.1f} ({clock(seconds)})".rjust(width)

with open(DATA_DIR / f"{SERIES}_selection_{EPISODE_COUNT}.csv",
          encoding="utf-8-sig", newline="") as f:
    SPLIT = {r["episode"]: r["split"] for r in csv.DictReader(f, delimiter=";")}
VIDEO_FILE = {ep: f"{SERIES}_{ep}.mp4" for ep in SPLIT}

_EPISODE = re.compile(r"s\d{2}e\d{2}", re.IGNORECASE)
_video_csv = WORK_DIR / f"{SERIES}_video.csv"
DURATION = {}
if _video_csv.exists():
    with open(_video_csv, encoding="utf-8-sig", newline="") as f:
        DURATION = {m.group(0).lower(): float(r["duration"])
                    for r in csv.DictReader(f, delimiter=";")
                    if (m := _EPISODE.search(r["file"]))}

# the verified annotation files, the register and the tag file are READ here, so
# that any cell below can be run on its own
verified, missing, broken = {}, [], {}
for ep in sorted(SPLIT):
    path = ANNOTATIONS_DIR / iv.file_name(SERIES, ep)
    if not path.exists():
        missing.append(ep)
        continue
    items = iv.read(path)
    problems = iv.validate(items, SERIES, ep)
    (broken if problems else verified)[ep] = problems if problems else items

rows = reg.load(REGISTER_CSV) if REGISTER_CSV.exists() else []
tag_rows = tags.load(TAGS_CSV)

print(f"{SERIES}: {len(SPLIT)} episodes selected, {len(verified)} verified,"
      f" {len(missing)} not annotated yet, {len(broken)} with problems")
print(f"register {len(rows)} rows, tag file {len(tag_rows)} rows"
      f"{'' if TAGS_CSV.exists() else ' (no ' + TAGS_CSV.name + ' yet)'}"
      f", durations for {len(DURATION)} episodes")

## 1. Wczytanie i kontrola zweryfikowanych plików

Każdy plik przechodzi tę samą kontrolę, którą robi adnotator: identyfikatory zgodne z konwencją i niepowtarzalne, dodatni czas trwania, brak nachodzenia adnotacji na maskę. Plik z błędem nie wchodzi do dalszych kroków.

In [ ]:
header = f"{'episode':<9}{'split':<7}{'annotations':>13}{'masks':>8}{'file'}"
print(header)
print("-" * (len(header) + 20))
for ep, items in verified.items():
    print(f"{ep:<9}{SPLIT[ep]:<7}"
          f"{sum(1 for i in items if not i.is_mask):>13}"
          f"{sum(1 for i in items if i.is_mask):>8}  {iv.file_name(SERIES, ep)}")
    if not any(i.is_mask for i in items):
        print(f"{'':<9}WARNING: no mask - the whole episode would enter the corpus")

if missing:
    print(f"\nnot verified yet ({len(missing)}): {', '.join(missing)}")
    print("run tbbt_02_annotation_candidates for them, then check them in the annotator")
for ep, problems in broken.items():
    print(f"\nFILE WITH PROBLEMS {ep} ({len(problems)}) - skipped:")
    for p in problems[:8]:
        print(f"   {p}")

print(f"\nverified {len(verified)} of {len(SPLIT)} episodes")
if REQUIRE_ALL and (missing or broken):
    raise SystemExit("REQUIRE_ALL = True and not every episode is ready")

## 2. Aktualizacja rejestru zbiorczego

**Zapisuje:** `data/interim/tbbt/tbbt_annotations.csv` - rejestr z naniesionymi decyzjami z adnotatora.

Porównanie zweryfikowanych plików `_intervals.csv` z rejestrem, wiersz po wierszu:

| co zrobiłaś w adnotatorze | co zapisuje rejestr |
|---|---|
| usunęłaś wiersz odrzucony jako `too_short` / `too_long` | nic, powód automatyczny zostaje |
| usunęłaś wiersz `accepted` | `status = rejected`, `reason = manual` |
| przesunęłaś czasy, długość bez zmian | tylko nowe `start` / `end`, bez flagi |
| zmieniłaś długość | nowe czasy, `edited_duration = yes`, werdykt liczony od nowa |
| poprawiłaś treść | nowa treść, `edited_desc = yes` |
| dopisałaś adnotację | nowy wiersz, `origin = manual` |

Rozdzielenie długości od treści jest po to, żeby dało się je policzyć osobno: poprawka literówki to co innego niż przesunięcie granicy zdarzenia. Samo przesunięcie nie stawia żadnej flagi, bo nie może zmienić werdyktu; widać je jako różnicę względem `source_start` i `source_end`, które pamiętają, co zapisał krok poprzedni.

Liczniki w tabeli: `moved` (ruszone czasy), `recovered` (wróciła po poprawieniu długości), `demoted` (wypadła po rozciągnięciu), `restored` (wróciła po ręcznym usunięciu), `added` (dopisana w adnotatorze).

Blok wypisuje też adnotacje leżące w całości w masce - to ostrzeżenie, nie werdykt. Taka adnotacja nie trafi w żaden fragment, więc jej zapytanie dostałoby pusty zbiór poprawnych odpowiedzi; usuwa się ją w adnotatorze.

In [ ]:
# A change of LENGTH is measured again, in both directions: a row rejected on its
# length comes back once it is repaired, and one stretched past the limit leaves.
# A pure shift needs no re-measuring. Length is CONTENT length -- a transition
# mask inside the annotation does not count -- as everywhere else.
_axis = {}
for _ep, _items in verified.items():
    _structural, _holes = rg.split_masks([i for i in _items if i.is_mask])
    _axis[_ep] = Timeline(rg.from_masks(_structural, DURATION[_ep]), _holes)

def revalidate(episode, start, end):
    length = _axis[episode].content_length(start, end)
    if length < settings.MIN_DURATION:
        return reg.REASON_TOO_SHORT
    if length > settings.MAX_DURATION:
        return reg.REASON_TOO_LONG
    return ""

rows = reg.load(REGISTER_CSV)
summary = reg.update(rows, verified, SPLIT, VIDEO_FILE, revalidate)
reg.save(rows, REGISTER_CSV)

COUNTS = ("kept", "removed", "moved", "edited_duration", "edited_desc",
          "recovered", "demoted", "restored", "added")

header = f"{'episode':<9}{'split':<7}" + "".join(f"{c:>16}" for c in COUNTS)
print(header)
print("-" * len(header))
for ep in sorted(summary):
    c = summary[ep]
    print(f"{ep:<9}{SPLIT[ep]:<7}" + "".join(f"{c[name]:>16}" for name in COUNTS))
print("-" * len(header))
print(f"{'total':<9}{'':<7}"
      + "".join(f"{sum(c[name] for c in summary.values()):>16}" for name in COUNTS))

# an annotation with no content left lies entirely inside a mask: it can never
# match a fragment, so its query would get an empty answer set
in_mask = [(ep, i.id) for ep, items in verified.items() for i in items
           if not i.is_mask and _axis[ep].content_length(i.start, i.end) <= 0]
if in_mask:
    print(f"\nWARNING: {len(in_mask)} annotations lie entirely inside a mask and can"
          " never match a fragment - remove them in the annotator:")
    for ep, annotation_id in in_mask[:10]:
        print(f"   {ep}  {annotation_id}")

removed = [r for r in rows if r["reason"] == reg.REASON_MANUAL
           and r["episode"] in verified]
print(f"\nrejected by hand ({len(removed)}) - check that this is what you meant:")
for r in removed[:20]:
    print(f"   {r['annotation_id']:<24}{r['start_mmss']:>8}  {r['desc'][:60]}")
if len(removed) > 20:
    print(f"   ... and {len(removed) - 20} more, all of them in {REGISTER_CSV.name}")

print(f"\nsaved -> {REGISTER_CSV.relative_to(ROOT)}")

## 3. Zakresy korpusu i dziury osi treści

**Zapisuje:** `data/interim/tbbt/tbbt_ranges.csv` - przedziały odcinka wchodzące do korpusu, oraz `data/interim/tbbt/tbbt_holes.csv` - maski przejścia wchłaniane przez oś treści. Oba pliki czyta potok przez `src/data/ranges.py::load_timelines` i nigdzie indziej nie odtwarza ich z masek.

**Dwie klasy masek robią co innego.** Strukturalne (logo, czołówka, napisy końcowe) tną odcinek na zakresy korpusu. Przejścia, czyli animacja atomu, niczego nie tną: stają się dziurami osi treści, więc zakres biegnie przez nie, a ich sekundy przestają się liczyć. Gdyby zakresy budować ze wszystkich masek, wariant E1-A dostałby za darmo kilkanaście poprawnych granic scen na odcinek, dokładnie tych, których sam znaleźć nie potrafi.

Kolumna `content` w tabeli to czas korpusu po odjęciu przejść; to on decyduje o długości fragmentu i o korekcie 3-15 s.

Czasy podawane są w sekundach, a w nawiasie ten sam czas jako `mm:ss` (wiersze odcinków) albo `hh:mm:ss` (sumy): sekundy do przepisania, zegar do sprawdzenia w odtwarzaczu. Wiersze sum na dole podają ten sam komplet kolumn co odcinki - materiał, korpus, treść i to, co odpadło.

In [ ]:
RANGES_CSV = DATA_DIR / f"{SERIES}_ranges.csv"
HOLES_CSV  = DATA_DIR / f"{SERIES}_holes.csv"

# The two classes of mask do different things and the split decides both files.
# Structural masks (logo, titles, credits) CUT the episode into corpus ranges.
# Transitions do not cut anything -- they become holes of the content axis, so
# their seconds stop counting while the range runs across them. Building the
# ranges from every mask would hand the fixed-window baseline of E1 a dozen free
# scene boundaries per episode.
range_rows, hole_rows, spans, holes = [], [], {}, {}
for ep, items in verified.items():
    structural, transitions = rg.split_masks([i for i in items if i.is_mask])
    spans[ep] = rg.from_masks(structural, DURATION[ep], MIN_RANGE)
    holes[ep] = transitions
    range_rows += rg.rows(ep, SPLIT[ep], VIDEO_FILE[ep], spans[ep])
    hole_rows += rg.hole_rows(ep, SPLIT[ep], VIDEO_FILE[ep], transitions)

for path, columns, table in ((RANGES_CSV, rg.COLUMNS, range_rows),
                             (HOLES_CSV, rg.HOLE_COLUMNS, hole_rows)):
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=columns, delimiter=";")
        writer.writeheader()
        writer.writerows(table)

header = (f"{'episode':<9}{'split':<7}{'ranges':>7}{'holes':>7}"
          f"{'file':>18}{'in corpus':>18}{'content':>18}{'dropped':>18}")
print(header)
print("-" * len(header))
for ep in sorted(spans, key=lambda e: (settings.split_order(SPLIT[e]), e)):
    inside = rg.total(spans[ep])
    content = inside - sum(e - s for s, e in holes[ep])
    print(f"{ep:<9}{SPLIT[ep]:<7}{len(spans[ep]):>7}{len(holes[ep]):>7}"
          + duration_cell(DURATION[ep]) + duration_cell(inside) + duration_cell(content)
          + duration_cell(DURATION[ep] - inside))
print("-" * len(header))

# sums per split and for everything, in hours
for group in [*settings.SPLITS, None]:
    chosen = [ep for ep in spans if group is None or SPLIT[ep] == group]
    if not chosen:
        continue
    material = sum(DURATION[ep] for ep in chosen)
    inside = sum(rg.total(spans[ep]) for ep in chosen)
    content = inside - sum(e - s for ep in chosen for s, e in holes[ep])
    label = group or "total"
    print(f"{label:<9}{f'{len(chosen)} odc.':<7}"
          f"{sum(len(spans[ep]) for ep in chosen):>7}"
          f"{sum(len(holes[ep]) for ep in chosen):>7}"
          + duration_cell(material, hhmmss) + duration_cell(inside, hhmmss)
          + duration_cell(content, hhmmss) + duration_cell(material - inside, hhmmss))

material = sum(DURATION[ep] for ep in spans)
inside = sum(rg.total(s) for s in spans.values())
print(f"\ndropped by the structural masks: {100 * (material - inside) / material:.1f}%"
      f" of the material; transitions inside the corpus: {len(hole_rows)}")
print(f"saved -> {RANGES_CSV.relative_to(ROOT)}, {HOLES_CSV.relative_to(ROOT)}")

## 4. Znaczniki zapytań

**Zapisuje:** `data/annotations/tbbt/tbbt_query_tags_new.csv` - jeden wiersz na przyjętą adnotację, z kolumnami na pięć znaczników wymagań, etykietę złożoności i wykaz postaci. Plik nie jest nadpisywany: blok dopisuje brakujące wiersze i odświeża kolumny kontekstowe (odcinek, split, treść zapytania), ale nigdy nie rusza komórki, która została wypełniona.

Znaczniki są pracą ręczną i dlatego mieszkają w osobnym pliku, a nie w gotowych plikach zapytań: te powstają od nowa przy każdym uruchomieniu tego notatnika, więc cokolwiek wpisanego wprost w nie zostałoby skasowane. Szkielet pliku dopisuje też skrypt `python scripts/prepare_data/query_tags.py tbbt --init --check`, który nigdy nie rusza wypełnionej komórki.

Znaczniki oznaczone w adnotatorze na samej adnotacji (kolumna `tags`) są tu wciągane. Odcinek, w którym dany znacznik został choć raz użyty, liczy się jako przejrzany pod jego kątem i pozostałe jego adnotacje dostają `0`. Odcinek, w którym znacznik nie pada ani razu, zostaje pusty - brak przejrzenia nie może wyglądać jak przejrzenie bez trafień.

**Postacie z profilami.** `PROFILED_CHARACTERS` wymienia postacie, dla których zbudowano profil tożsamości. Lista rządzi dwiema rzeczami przy budowie plików zapytań: z `identities` znikają osoby bez profilu, a znacznik `wymaga_osoby` jest przeliczany na to, czy cokolwiek zostało. Imię bez profilu nie może uruchomić sygnału tożsamości, więc pozostawienie go obiecywałoby komponent, który i tak się nie włączy.

Kolejną postać dopisuje się, gdy plik znaczników będzie wypełniony: blok statystyki niżej wypisze wtedy, które imiona wracają najczęściej i w ilu odcinkach.

In [ ]:
PROFILED_CHARACTERS = settings.profiled("tbbt")

TAGS_CSV = tags.path_for(SERIES, ANNOTATIONS_DIR)
TAGS_NEW = TAGS_CSV.with_name(TAGS_CSV.stem + "_new.csv")
# only episodes that went through the annotator: tagging an annotation
# that verification may still delete or reword is work done twice
accepted = reg.accepted([r for r in rows if r["episode"] in verified])

tag_rows, tag_counts = tags.skeleton(accepted, tags.load(TAGS_CSV),
                                     tags.from_register(accepted))
tags.save(tag_rows, TAGS_NEW)

annotator = tags.annotator_report(accepted)
marked = {tag: n for tag, n in annotator["marked"].items() if n}
print("tags marked in the annotator:", marked or "none yet")
if annotator["unknown"]:
    print("labels outside the vocabulary (ignored):", ", ".join(annotator["unknown"]))

problems = tags.validate(tag_rows)
for problem in problems[:10]:
    print(f"  INVALID {problem}")
if problems:
    print(f"{len(problems)} invalid cells - fix them before the queries are rebuilt")

print(f"\nrows: {len(tag_rows)}  (added {tag_counts['added']},"
      f" kept {tag_counts['kept']}, dropped {tag_counts['dropped']},"
      f" from the annotator {tag_counts['from_annotator']})")
if tag_counts["retexted"]:
    print(f"WARNING: wording changed for {len(tag_counts['retexted'])} rows -"
          " their tags were assigned to a different query")

header = f"{'episode':<10}{'split':<7}{'filled':>8}{'total':>8}"
print(header)
print("-" * len(header))
for split in ("dev", "test"):
    stats = tags.coverage(tag_rows, split)
    for episode, entry in stats["per_episode"].items():
        print(f"{episode:<10}{split:<7}{entry['filled']:>8}{entry['total']:>8}")
    if stats["total"]:
        print(f"{'total ' + split:<10}{'':<7}{stats['filled']:>8}{stats['total']:>8}")
print(f"saved -> {TAGS_NEW.relative_to(ROOT)}")
print(f"copy it over {TAGS_CSV.name} by hand once you are done")

## 5. Kontrola kompletności znaczników

Sprawdza plik, z którego naprawdę powstają zapytania, czyli `data/annotations/tbbt/tbbt_query_tags.csv`, a nie szkielet `_new` zapisany w bloku wyżej. Wiersz jest kompletny, gdy ma wartość w każdym znaczniku wymagań i w kolumnie złożoności; `identities` do tego nie wchodzi, bo puste jest tam poprawną odpowiedzią (`tags.is_filled`).

Wypisuje odcinki, którym czegoś brakuje, wraz z nazwą kolumny i liczbą pustych komórek. Kolumna pusta w całym odcinku znaczy, że odcinek nie został pod tym kątem przejrzany, a to co innego niż przejrzenie bez trafień, dlatego brak i zero muszą wyglądać inaczej.

**Kontrola niczego nie blokuje.** Zapytania powstają także z niekompletnym plikiem: brakujący znacznik zostawia puste pole w rekordzie zapytania i wypada z podzbiorów raportowanych, ale nie zmienia ani rankingu, ani metryk.

Wierszy, które straciły swoją adnotację, ten blok nie liczy; robi to krok wyżej, jako `dropped` w podsumowaniu szkieletu.

In [ ]:
# The tag file is read HERE, not taken from the configuration cell: after filling
# it in by hand you run this cell alone, and it has to see what is in the file now.
tag_rows = tags.load(TAGS_CSV)
DECISION_COLUMNS = tags.tags_in(tags.COLUMNS) + ["complexity"]

if not tag_rows:
    print(f"no {TAGS_CSV.name} yet - nothing to check")
else:
    header = f"{'episode':<10}{'split':<7}{'complete':>10}{'rows':>7}   missing"
    print(header)
    print("-" * len(header))
    short = 0
    for split in SPLITS:
        for episode, entry in tags.coverage(tag_rows, split)["per_episode"].items():
            if entry["filled"] == entry["total"]:
                continue
            short += 1
            here = [r for r in tag_rows.values()
                    if r.get("episode") == episode and r.get("split") == split]
            gaps = [(column, sum(1 for r in here if not (r.get(column) or "").strip()))
                    for column in DECISION_COLUMNS]
            named = ", ".join(f"{c} ({n})" for c, n in gaps if n)
            print(f"{episode:<10}{split:<7}{entry['filled']:>10}{entry['total']:>7}   {named}")

    whole = tags.coverage(tag_rows)
    print(f"\n{whole['filled']} of {whole['total']} rows complete in {TAGS_CSV.name}")
    print("every episode carries a complete set" if not short else
          f"{short} episode(s) short of one - the queries below are written anyway,"
          " without the tags that are missing")

## 6. Pliki zapytań

**Zapisuje:** `data/annotations/tbbt/tbbt_queries_dev.jsonl` i `tbbt_queries_test.jsonl` - po jednym pliku na zbiór, w schemacie wspólnym dla wszystkich trzech zbiorów danych (`src/utils/queries.py`). Oba powstają zawsze, także gdy któryś zbiór jest jeszcze pusty, więc brakujący plik zawsze znaczy, że tego kroku nie uruchomiono.

Wchodzą wyłącznie adnotacje przyjęte. Dla serialu jednostką wyszukiwania jest fragment odcinka, więc `vid_name` wskazuje nagranie (`tbbt_s01e01`), a `ts` to granice zdarzenia na jego osi czasu. Każda adnotacja jest osobnym zdarzeniem, więc `event_id` bierze identyfikator adnotacji, a ten wciąż niesie `desc_id` z TVR i prowadzi z powrotem do źródła.

Blok czyta plik znaczników z dysku, nie z poprzedniego bloku, więc po ręcznym uzupełnieniu `<zbiór>_query_tags.csv` można uruchomić sam ten. Gdy pliku jeszcze nie ma, zapytania powstają bez znaczników i blok to wypisuje.

Do plików trafiają wyłącznie odcinki, które przeszły przez adnotator - ta sama reguła co w pliku znaczników. Zapytanie zbudowane z niezweryfikowanej adnotacji niosłoby czasy, na które nikt nie patrzył, więc część testowa zostaje pusta do czasu zweryfikowania jej odcinków.

In [ ]:
PROFILED_CHARACTERS = settings.profiled("tbbt")

# The tag file is read HERE, not taken from the cell above: after filling it in
# by hand you run this cell alone, and it has to see what is in the file now.
TAGS_CSV = tags.path_for(SERIES, ANNOTATIONS_DIR)
tag_rows = tags.load(TAGS_CSV)

# Only episodes that went through the annotator, exactly as in the tag file: a
# query built from an unverified annotation would carry times nobody has looked
# at, so the test split stays empty until its episodes are verified.
verified_rows = [r for r in rows if r["episode"] in verified]
if not tag_rows:
    print(f"no {TAGS_CSV.name} yet - queries are written without tags;"
          " copy the _new file over it and run this cell again\n")

written = build.write_splits(verified_rows, ANNOTATIONS_DIR, SERIES, tag_rows=tag_rows,
                             profiled=PROFILED_CHARACTERS)

header = f"{'split':<7}{'episodes':>10}{'queries':>9}{'median [s]':>13}{'total [s]':>12}"
print(header)
print("-" * len(header))
for s in build.summary(verified_rows):
    print(f"{s['split']:<7}{s['episodes']:>10}{s['queries']:>9}"
          f"{s['median_duration']:>13.2f}{s['total_duration']:>12.1f}")

for split, path in written.items():
    print(f"saved -> {path.relative_to(ROOT)}")

from src.utils.queries import load_jsonl

example = load_jsonl(written["test"])
print(f"\nexample record:\n{example[0].to_dict() if example else '(no queries yet)'}")

## 7. Statystyka zapytań

Osobno dla części deweloperskiej i testowej. Liczebności rozstrzygają, które przekroje da się analizować: znacznik występujący w kilkunastu zapytaniach nie utrzyma przedziału ufności i wchodzi dalej jako wielkość efektu.

Wiersze z postaciami liczą, ile zapytań przywołuje jedną postać z profilem, ile dwie, ile trzy. Zapytania bez żadnej postaci z profilem są policzone osobno - dla nich sygnał tożsamości jest nieaktywny.

In [ ]:
from src.annotation import build as bld
from src.evaluation import tables

SPLITS = ("dev", "test")

per_split = {split: tags.statistics(
                 bld.queries(reg.accepted([r for r in rows if r["episode"] in verified],
                                          split), SERIES,
                             tag_rows, PROFILED_CHARACTERS))
             for split in SPLITS}

def share(stats, value):
    """Count with its share of the split, or "-" when the split is empty."""
    if not stats["count"]:
        return "-"
    return f"{value:>4}  ({100 * value / stats['count']:>4.1f}%)"


table = [["liczba zapytan"] + [str(per_split[s]["count"]) for s in SPLITS]]
for tag in tags.REQUIREMENT_TAGS:
    table.append([tag] + [share(per_split[s], per_split[s]["requirements"][tag])
                         for s in SPLITS])
for value, name in (("P", "proste (P)"), ("Z", "zlozone (Z)")):
    table.append([name] + [share(per_split[s], per_split[s]["complexity"][value])
                          for s in SPLITS])

# how many queries name one profiled character, how many two, how many three
counts = sorted({n for s in per_split.values() for n in s["identities"]})
for many in counts:
    name = ("bez postaci z profilem" if many == 0
            else "1 postac z profilem" if many == 1
            else f"{many} postaci z profilami")
    table.append([name] + [share(per_split[s], per_split[s]["identities"].get(many, 0))
                          for s in SPLITS])

tables.show(f"Zapytania {SERIES} wedlug znacznikow",
            ["Wielkosc", "dev", "test"], table,
            note="Udzial liczony wzgledem liczby zapytan danej czesci. Wiersze "
                 "z postaciami dotycza wylacznie postaci majacych profil - to one "
                 "decyduja, dla ktorych zapytan sygnal tozsamosci jest aktywny.")

# which names recur at all - the input to choosing PROFILED_CHARACTERS
seen = {}
for row in tag_rows.values():
    for name in tags.identities_of(row):
        entry = seen.setdefault(name, {"queries": 0, "episodes": set()})
        entry["queries"] += 1
        entry["episodes"].add(row["episode"])
if seen:
    order = sorted(seen.items(),
                   key=lambda kv: (-len(kv[1]["episodes"]), -kv[1]["queries"]))
    print()
    print("imiona w identities, wg kryterium doboru profili z rozdzialu 4:")
    print(f"  {'imie':<14}{'odcinkow':>9}{'zapytan':>9}   profil")
    for name, entry in order:
        mark = "tak" if tags.resolve_characters([name], PROFILED_CHARACTERS) else "-"
        print(f"  {name:<14}{len(entry['episodes']):>9}"
              f"{entry['queries']:>9}   {mark}")

missing_label = sum(s["complexity_missing"] for s in per_split.values())
if missing_label:
    print()
    print(f"zapytan bez etykiety zlozonosci: {missing_label}"
          " - uzupelnij plik znacznikow")

## 8. Statystyki adnotacji

Ile adnotacji miał każdy odcinek na wejściu i co się z nimi stało. Powody odrzucenia: `too_short` (krócej niż krok próbkowania klatek), `too_long` (dłużej niż jeden fragment), `manual` (usunięte ręcznie w adnotatorze). Kolumny `edited_duration` i `edited_desc` liczą adnotacje, którym w adnotatorze zmieniono odpowiednio długość i treść opisu (wartość `yes` w rejestrze `<seria>_annotations.csv`).

In [ ]:
header = (f"{'episode':<9}{'split':<7}{'total':>7}{'accepted':>10}"
          + "".join(f"{r:>11}" for r in reg.REASONS) + f"{'kept %':>9}"
          + "".join(f"{e:>{len(e) + 2}}" for e in reg.EDITS))
print(header)
print("-" * len(header))
for s in reg.stats(rows):
    share = 100 * s["accepted"] / s["total"] if s["total"] else 0.0
    print(f"{s['episode']:<9}{s['split']:<7}{s['total']:>7}{s['accepted']:>10}"
          + "".join(f"{s[r]:>11}" for r in reg.REASONS) + f"{share:>9.1f}"
          + "".join(f"{s[e]:>{len(e) + 2}}" for e in reg.EDITS))
print("-" * len(header))
for t in reg.totals(rows):
    share = 100 * t["accepted"] / t["total"] if t["total"] else 0.0
    print(f"{t['split']:<9}{'':<7}{t['total']:>7}{t['accepted']:>10}"
          + "".join(f"{t[r]:>11}" for r in reg.REASONS) + f"{share:>9.1f}"
          + "".join(f"{t[e]:>{len(e) + 2}}" for e in reg.EDITS))

def _same_times(r):
    return ((r.get("source_start"), r.get("source_end"))
            == (r.get("start"), r.get("end")))

moved = sum(1 for r in rows if not _same_times(r))
manual = sum(1 for r in rows if r.get("origin") == reg.ORIGIN_MANUAL)
uncertain = sum(1 for r in rows if reg.FLAG_UNCERTAIN in (r.get("flags") or ""))
print(f"\ntimes differing from the ones the register was built with: {moved}")
print("  the length edits are counted in the table (edited_duration); the rest")
print("  is the fitting to the masks plus hand shifts, detailed in")
print(f"  work/{SERIES}_fitting_<variant>.csv")
print(f"annotations created in the annotator:                       {manual}")
print(f"uncertain_mapping:                                          {uncertain}")